In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import numpy as np
import seaborn as sns
import math
myplate = sns.color_palette("Paired")
sns.set(style='white', palette=myplate)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import warnings
warnings.filterwarnings("ignore")
import joblib
clrs = ["#ADD8E6","#FF0000"]
mypalette = sns.set_palette(sns.color_palette(clrs))

# EDA

In [ ]:
sal_path = "/kaggle/input/jobs-dataset-from-glassdoor/eda_data.csv"
sal = pd.read_csv(sal_path, index_col=0)
sal.head()

In [ ]:
sal.describe()

In [ ]:
sal.shape

In [ ]:
sal.columns

In [ ]:
colonnes = ['job_simp', 'Rating', 'Company Name', 'Size', 'Founded', 'Industry',
            'Sector', 'min_salary', 'avg_salary', 'max_salary', 'age', 'Type of ownership']
df = sal[colonnes].copy()

In [ ]:
print(df)

In [ ]:
# 1. RENOMMER LES COLONNES
df.columns = df.columns.str.lower().str.replace(' ', '_')
print("✓ Colonnes renommées")

# 2. NETTOYAGE
df['company_name'] = df['company_name'].str.replace(r'\n\d+\.\d+', '', regex=True)
df = df.replace('na', np.nan).replace(-1, np.nan)
print("✓ Données nettoyées")

# 3. IDENTIFIER LES COLONNES CATÉGORIELLES ET NUMÉRIQUES
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Colonnes catégorielles: {categorical_cols}")
print(f"Colonnes numériques: {numerical_cols}")

# 4. GÉRER LES VALEURS MANQUANTES
print(f"Avant: {df.isnull().sum().sum()} valeurs manquantes")
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna('unknown', inplace=True)
print(f"Après: {df.isnull().sum().sum()} valeurs manquantes")

In [ ]:
df["rating"] = df["rating"].round().astype("Int64")
df["founded"] = df["founded"].round().astype("Int64")
df["avg_salary"] = df["avg_salary"].round().astype("Int64")
df["age"] = df["avg_salary"].round().astype("Int64")

## Analyse des données

In [ ]:
print("\n1. Distribution des salaires par type de poste")
plt.figure(figsize=(14, 8))
df_sorted = df.groupby('job_simp')['avg_salary'].median().sort_values(ascending=False)
top_jobs = df_sorted.head(10).index
sns.boxplot(data=df[df['job_simp'].isin(top_jobs)],
            x='avg_salary',
            y='job_simp',
            order=top_jobs,
            palette='viridis')
plt.title('Distribution des Salaires par Type de Poste (Top 10)', fontsize=16, fontweight='bold')
plt.xlabel('Salaire Moyen (k$)', fontsize=12)
plt.ylabel('Type de Poste', fontsize=12)
plt.tight_layout()
plt.savefig('salary_distribution_by_job.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: salary_distribution_by_job.png")
plt.show()

In [ ]:
print("\n2. Matrice de corrélation des variables numériques...")
plt.figure(figsize=(12, 10))
numeric_cols = ['rating', 'min_salary', 'avg_salary', 'max_salary', 'age']
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title('Matrice de Corrélation des Variables Numériques', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: correlation_matrix.png")
plt.show()

In [ ]:
print("\n3. Répartition des emplois par secteur...")
plt.figure(figsize=(14, 8))
sector_counts = df['sector'].value_counts().head(10)
plt.barh(range(len(sector_counts)), sector_counts.values, color='steelblue')
plt.yticks(range(len(sector_counts)), sector_counts.index)
plt.xlabel("Nombre d\'Offres", fontsize=12)
plt.ylabel('Secteur', fontsize=12)
plt.title("Top 10 des Secteurs avec le Plus d\'Offres d\'Emploi", fontsize=16, fontweight='bold')
for i, v in enumerate(sector_counts.values):
    plt.text(v + 1, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('jobs_by_sector.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: jobs_by_sector.png")
plt.show()

In [ ]:
print("\n4. Relation entre l'âge de l'entreprise et le salaire...")
plt.figure(figsize=(12, 7))
df['age_bin'] = pd.cut(df['age'], bins=[0, 10, 20, 50, 100, 200],
                        labels=['0-10 ans', '11-20 ans', '21-50 ans', '51-100 ans', '100+ ans'])
sns.boxplot(data=df, x='age_bin', y='avg_salary', palette='Set2')
plt.title("Salaire Moyen selon l\'Âge de l\'Entreprise", fontsize=16, fontweight='bold')
plt.xlabel("Âge de l\'Entreprise", fontsize=12)
plt.ylabel('Salaire Moyen (k$)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('salary_by_company_age.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: salary_by_company_age.png")
plt.show()

In [ ]:
print("\n5. Distribution des ratings des entreprises...")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hist(df['rating'], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(df['rating'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Moyenne: {df["rating"].mean():.2f}')
axes[0].axvline(df['rating'].median(), color='green', linestyle='--', linewidth=2,
                label=f'Médiane: {df["rating"].median():.2f}')
axes[0].set_xlabel('Rating', fontsize=12)
axes[0].set_ylabel('Fréquence', fontsize=12)
axes[0].set_title('Distribution des Ratings', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df['rating'], df['avg_salary'], alpha=0.5, s=50, color='coral')
axes[1].set_xlabel('Rating', fontsize=12)
axes[1].set_ylabel('Salaire Moyen (k$)', fontsize=12)
axes[1].set_title('Relation Rating vs Salaire', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

z = np.polyfit(df['rating'].dropna(), df.loc[df['rating'].notna(), 'avg_salary'], 1)
p = np.poly1d(z)
axes[1].plot(df['rating'].sort_values(), p(df['rating'].sort_values()),
             "r--", linewidth=2, label='Tendance')
axes[1].legend()
plt.tight_layout()
plt.savefig('rating_analysis.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: rating_analysis.png")
plt.show()

In [ ]:
print("\n6. Salaire moyen par taille d'entreprise...")
plt.figure(figsize=(12, 7))
size_order = ['unknown', '1 to 50 employees', '51 to 200 employees',
              '201 to 500 employees', '501 to 1000 employees',
              '1001 to 5000 employees', '5001 to 10000 employees', '10000+ employees']
available_sizes = [s for s in size_order if s in df['size'].unique()]
sns.violinplot(data=df[df['size'].isin(available_sizes)],
               x='size',
               y='avg_salary',
               order=available_sizes,
               palette='muted')
plt.xticks(rotation=45, ha='right')
plt.title("Distribution des Salaires par Taille d\'Entreprise", fontsize=16, fontweight='bold')
plt.xlabel("Taille de l\'Entreprise", fontsize=12)
plt.ylabel('Salaire Moyen (k$)', fontsize=12)
plt.tight_layout()
plt.savefig('salary_by_company_size.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: salary_by_company_size.png")
plt.show()

In [ ]:
print("\n7. Salaires moyens par secteur...")
plt.figure(figsize=(14, 8))
top_sectors = df.groupby('sector')['avg_salary'].mean().sort_values(ascending=False).head(10)
colors_s = plt.cm.viridis(np.linspace(0, 1, len(top_sectors)))
plt.barh(range(len(top_sectors)), top_sectors.values, color=colors_s)
plt.yticks(range(len(top_sectors)), top_sectors.index)
plt.xlabel('Salaire Moyen (k$)', fontsize=12)
plt.ylabel('Secteur', fontsize=12)
plt.title('Top 10 des Secteurs les Mieux Rémunérés', fontsize=16, fontweight='bold')
for i, v in enumerate(top_sectors.values):
    plt.text(v + 1, i, f'{v:.1f}k$', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('top_sectors_by_salary.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: top_sectors_by_salary.png")
plt.show()

## Préparation pour l'entrainement

In [ ]:
if 'company_name' in categorical_cols:
    categorical_cols.remove('company_name')

label_encoders = {}
for col in categorical_cols:
    print(f"Encodage de '{col}': {df[col].nunique()} catégories uniques")
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

In [ ]:
feature_cols = []
numeric_features = [col for col in numerical_cols if col not in ['avg_salary']]
feature_cols.extend(numeric_features)
encoded_cols = [col + '_encoded' for col in categorical_cols]
feature_cols.extend(encoded_cols)
print(f"Features sélectionnées ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
X = df[feature_cols].copy()
y = df['avg_salary'].copy()
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
print("VÉRIFICATION DES TYPES DE DONNÉES")
print(X.dtypes)

In [ ]:
non_numeric = X.select_dtypes(include=['object']).columns
if len(non_numeric) > 0:
    print(f"ATTENTION: Colonnes non-numériques détectées: {non_numeric.tolist()}")
    for col in non_numeric:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        X[col].fillna(X[col].median(), inplace=True)
else:
    print("Toutes les colonnes sont numériques")

In [ ]:
print("NORMALISATION DES DONNÉES")
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

In [ ]:
print("CRÉATION DES SETS TRAIN/TEST")
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")
print("\n✓ Fichiers sauvegardés!")
print(f"Dataset original: {df.shape}")
print(f"Nombre de features: {X.shape[1]}")
print(f"Features: {X.columns.tolist()}")

## Entrainement des données

In [ ]:
print("Données chargées:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nTypes de données dans X_train:")
print(X_train.dtypes)

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}
results = {}

In [ ]:
for name, model in models.items():
    print(f"\n {name}...")
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    r2 = r2_score(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae = mean_absolute_error(y_test, y_pred_test)
    results[name] = {'R²': r2, 'RMSE': rmse, 'MAE': mae}
    print(f"   R²: {r2:.4f}")
    print(f"   RMSE: {rmse:.2f}")
    print(f"   MAE: {mae:.2f}")

print("COMPARAISON DES MODÈLES")
comparison = pd.DataFrame(results).T
print(comparison.sort_values('R²', ascending=False))

## Analyse des résultats

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}
results = {}
predictions = {}

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    predictions[name] = y_pred_test

    train_r2 = r2_score(y_train, y_pred_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)
    mape = np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100

    results[name] = {
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'MAE': test_mae,
        'MAPE (%)': mape,
        'Overfitting': train_r2 - test_r2
    }

    print(f"{name}")
    print(f"  R² Score (Train): {train_r2:.4f}")
    print(f"  R² Score (Test):  {test_r2:.4f}")
    print(f"  RMSE (Train):     {train_rmse:.2f}")
    print(f"  RMSE (Test):      {test_rmse:.2f}")
    print(f"  MAE (Test):       {test_mae:.2f}")
    print(f"  MAPE:             {mape:.2f}%")
    print(f"  Overfitting:      {train_r2 - test_r2:.4f}")
    if train_r2 - test_r2 > 0.05:
        print(f"  ⚠ Overfitting détecté!")
    else:
        print(f"  ✓ Bonne généralisation")

In [ ]:
print("TABLEAU COMPARATIF")
comparison_df = pd.DataFrame(results).T.round(4)
print(comparison_df.to_string())

best_model_name = comparison_df['Test R²'].idxmax()
print(f"\nMEILLEUR MODÈLE: {best_model_name}")
print(f"   R² Test: {comparison_df.loc[best_model_name, 'Test R²']:.4f}")
print(f"   RMSE: {comparison_df.loc[best_model_name, 'Test RMSE']:.2f}")

In [ ]:
print("CRÉATION DES VISUALISATIONS")
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics_df = pd.DataFrame({
    'Model': list(results.keys()),
    'R² Train': [results[m]['Train R²'] for m in results.keys()],
    'R² Test': [results[m]['Test R²'] for m in results.keys()]
})

ax = axes[0, 0]
x = np.arange(len(metrics_df))
width = 0.35
ax.bar(x - width/2, metrics_df['R² Train'], width, label='Train', alpha=0.8)
ax.bar(x + width/2, metrics_df['R² Test'], width, label='Test', alpha=0.8)
ax.set_ylabel('R² Score')
ax.set_title('Comparaison R² Score (Train vs Test)')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
rmse_data = [results[m]['Test RMSE'] for m in results.keys()]
colors_bar = ['#ff7675' if r == min(rmse_data) else '#74b9ff' for r in rmse_data]
ax.bar(results.keys(), rmse_data, color=colors_bar, alpha=0.8)
ax.set_ylabel('RMSE')
ax.set_title('RMSE sur le Test Set')
ax.set_xticklabels(results.keys(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
mae_data = [results[m]['MAE'] for m in results.keys()]
colors_bar = ['#00b894' if m == min(mae_data) else '#74b9ff' for m in mae_data]
ax.bar(results.keys(), mae_data, color=colors_bar, alpha=0.8)
ax.set_ylabel('MAE')
ax.set_title('Mean Absolute Error (MAE)')
ax.set_xticklabels(results.keys(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
mape_data = [results[m]['MAPE (%)'] for m in results.keys()]
colors_bar = ['#fdcb6e' if m == min(mape_data) else '#74b9ff' for m in mape_data]
ax.bar(results.keys(), mape_data, color=colors_bar, alpha=0.8)
ax.set_ylabel('MAPE (%)')
ax.set_title('Mean Absolute Percentage Error')
ax.set_xticklabels(results.keys(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_metrics_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Graphique 1 sauvegardé: model_metrics_comparison.png")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()
for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[idx]
    ax.scatter(y_test, y_pred, alpha=0.5, s=50)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Prédiction parfaite')
    r2 = results[name]['Test R²']
    rmse = results[name]['Test RMSE']
    ax.set_xlabel('Valeurs Réelles (avg_salary)', fontsize=10)
    ax.set_ylabel('Prédictions', fontsize=10)
    ax.set_title(f'{name}\nR² = {r2:.4f}, RMSE = {rmse:.2f}', fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('predictions_vs_real.png', dpi=300, bbox_inches='tight')
print("✓ Graphique sauvegardé: predictions_vs_real.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()
for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[idx]
    errors = y_test - y_pred
    ax.hist(errors, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Erreur = 0')
    ax.set_xlabel('Erreur de Prédiction')
    ax.set_ylabel('Fréquence')
    ax.set_title(f'{name}\nMoyenne: {errors.mean():.2f}, Std: {errors.std():.2f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('error_distributions.png', dpi=300, bbox_inches='tight')
print("✓ Graphique sauvegardé: error_distributions.png")
plt.show()

In [ ]:
print("IMPORTANCE DES FEATURES (Random Forest)")
rf_model = models['Random Forest']
rf_model.fit(X_train, y_train)
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance', fontsize=12)
plt.title('Importance des Features (Random Forest)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
print("✓ Graphique sauvegardé: feature_importance.png")
plt.show()

In [ ]:
best_model = models[best_model_name]
best_model.fit(X_train, y_train)
joblib.dump(best_model, f'best_model_{best_model_name.replace(" ", "_")}.pkl')
print(f"\n✓ Meilleur modèle sauvegardé: best_model_{best_model_name.replace(' ', '_')}.pkl")
print(f"\nRÉSUMÉ:")
print(f"   - Meilleur modèle: {best_model_name}")
print(f"   - R² Score: {comparison_df.loc[best_model_name, 'Test R²']:.4f}")
print(f"   - RMSE: {comparison_df.loc[best_model_name, 'Test RMSE']:.2f}")
print(f"   - MAE: {comparison_df.loc[best_model_name, 'MAE']:.2f}")
print(f"   - MAPE: {comparison_df.loc[best_model_name, 'MAPE (%)']:.2f}%")

## Isolation Forest

In [ ]:
print("=" * 60)
print("ISOLATION FOREST - DÉTECTION D'ANOMALIES")
print("=" * 60)

# Entraînement de l'Isolation Forest sur X_train
iso_forest = IsolationForest(contamination='auto', random_state=42)
iso_forest.fit(X_train)
print("✓ Isolation Forest entraîné sur X_train")

# Prédiction sur le jeu d'ENTRAÎNEMENT
train_predictions_if = iso_forest.predict(X_train)
train_scores_if = iso_forest.score_samples(X_train)

n_outliers_train = (train_predictions_if == -1).sum()
n_inliers_train = (train_predictions_if == 1).sum()
pct_outliers_train = 100 * n_outliers_train / len(train_predictions_if)

print(f"\nRésultats sur le jeu d'ENTRAÎNEMENT:")
print(f"  - Total observations  : {len(train_predictions_if)}")
print(f"  - Inliers  (normaux)  : {n_inliers_train} ({100 - pct_outliers_train:.2f}%)")
print(f"  - Outliers (anomalies): {n_outliers_train} ({pct_outliers_train:.2f}%)")
print(f"\nStatistiques des scores d'anomalie (train):")
print(f"  - Score min  : {train_scores_if.min():.4f}")
print(f"  - Score max  : {train_scores_if.max():.4f}")
print(f"  - Score moyen: {train_scores_if.mean():.4f}")
print(f"  - Score std  : {train_scores_if.std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(train_scores_if, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(train_scores_if.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Moyenne: {train_scores_if.mean():.3f}')
axes[0].set_xlabel("Score d'Anomalie (Isolation Forest)", fontsize=12)
axes[0].set_ylabel('Fréquence', fontsize=12)
axes[0].set_title("Distribution des Scores d'Anomalie\n(Jeu d'Entraînement)", fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

colors_if = ['red' if p == -1 else 'steelblue' for p in train_predictions_if]
axes[1].scatter(train_scores_if, y_train, c=colors_if, alpha=0.5, s=30)
axes[1].set_xlabel("Score d'Anomalie", fontsize=12)
axes[1].set_ylabel('avg_salary (k$)', fontsize=12)
axes[1].set_title("Score d'Anomalie vs Salaire Moyen\n(rouge = outlier)", fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Inlier (normal)'),
                   Patch(facecolor='red', label='Outlier (anomalie)')]
axes[1].legend(handles=legend_elements)

plt.tight_layout()
plt.savefig('isolation_forest_train_scores.png', dpi=300, bbox_inches='tight')
print("✓ Graphique sauvegardé: isolation_forest_train_scores.png")
plt.show()

In [ ]:
print("SCORES DE L'ISOLATION FOREST SUR LE JEU D'ENTRAÎNEMENT")
print("-" * 60)

mask_inliers = train_predictions_if == 1
mask_outliers = train_predictions_if == -1

print("Statistiques des salaires - INLIERS:")
print(f"  - Moyenne : {y_train[mask_inliers].mean():.2f} k$")
print(f"  - Médiane : {y_train[mask_inliers].median():.2f} k$")
print(f"  - Std     : {y_train[mask_inliers].std():.2f} k$")
print(f"  - Min     : {y_train[mask_inliers].min():.2f} k$")
print(f"  - Max     : {y_train[mask_inliers].max():.2f} k$")

print("\nStatistiques des salaires - OUTLIERS:")
print(f"  - Moyenne : {y_train[mask_outliers].mean():.2f} k$")
print(f"  - Médiane : {y_train[mask_outliers].median():.2f} k$")
print(f"  - Std     : {y_train[mask_outliers].std():.2f} k$")
print(f"  - Min     : {y_train[mask_outliers].min():.2f} k$")
print(f"  - Max     : {y_train[mask_outliers].max():.2f} k$")

# Modèle de référence pour la suite
best_model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
best_model_rf.fit(X_train, y_train)
y_pred_test_rf = best_model_rf.predict(X_test)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_test_rf))
print(f"\nRMSE de base (Random Forest, test complet): {rmse_base:.4f}")

In [ ]:
print("DÉFINITION DU SEUIL DE ROBUSTESSE (MÉTHODE DU COUDE)")
print("-" * 60)

test_scores_if = iso_forest.score_samples(X_test)
y_pred_all = best_model_rf.predict(X_test)

thresholds = np.linspace(test_scores_if.min(), test_scores_if.max(), 100)
count_pct = []
rmse_list = []
n_total = len(y_test)

for t in thresholds:
    mask = test_scores_if >= t
    count_pct.append(100 * mask.sum() / n_total)
    if mask.sum() > 1:
        error = mean_squared_error(y_test[mask], y_pred_all[mask])
        rmse_list.append(np.sqrt(error))
    else:
        rmse_list.append(np.nan)

# Méthode du coude
rmse_array = np.array(rmse_list)
valid_mask = ~np.isnan(rmse_array)
thresholds_valid = thresholds[valid_mask]
rmse_valid = rmse_array[valid_mask]
count_valid = np.array(count_pct)[valid_mask]

x_norm = (thresholds_valid - thresholds_valid.min()) / (thresholds_valid.max() - thresholds_valid.min())
y_norm = (rmse_valid - rmse_valid.min()) / (rmse_valid.max() - rmse_valid.min())

p1 = np.array([x_norm[0], y_norm[0]])
p2 = np.array([x_norm[-1], y_norm[-1]])
line_vec = p2 - p1
line_vec_norm = line_vec / np.linalg.norm(line_vec)

distances = []
for i in range(len(x_norm)):
    point = np.array([x_norm[i], y_norm[i]])
    vec_to_point = point - p1
    proj = np.dot(vec_to_point, line_vec_norm) * line_vec_norm
    dist = np.linalg.norm(vec_to_point - proj)
    distances.append(dist)

distances = np.array(distances)
elbow_idx = np.argmax(distances)
optimal_threshold = thresholds_valid[elbow_idx]
optimal_coverage = count_valid[elbow_idx]
optimal_rmse = rmse_valid[elbow_idx]

print(f"✓ Seuil optimal (coude)  : {optimal_threshold:.4f}")
print(f"✓ Coverage correspondant : {optimal_coverage:.1f}%")
print(f"✓ RMSE au seuil optimal  : {optimal_rmse:.4f}")
print(f"✓ RMSE de base           : {rmse_base:.4f}")
print(f"✓ Gain RMSE              : {rmse_base - optimal_rmse:.4f} ({100*(rmse_base - optimal_rmse)/rmse_base:.2f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax1 = axes[0]
ax1.plot(thresholds_valid, rmse_valid, color='tab:blue', linewidth=2, label='RMSE')
ax1.axvline(optimal_threshold, color='purple', linestyle=':', linewidth=2.5,
            label=f'Seuil optimal (coude): {optimal_threshold:.3f}')
ax1.scatter([optimal_threshold], [optimal_rmse], color='purple', s=120, zorder=5)
ax1.set_xlabel("Seuil de Score d'Isolation (plus élevé = plus strict)", fontsize=11)
ax1.set_ylabel('RMSE (Erreur)', color='tab:blue', fontsize=11)
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(thresholds_valid, count_valid, color='tab:red', linestyle='--', linewidth=2, label='Coverage %')
ax2.set_ylabel('Coverage (% des données conservées)', color='tab:red', fontsize=11)
ax2.tick_params(axis='y', labelcolor='tab:red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('Analyse de Robustesse : RMSE vs Coverage\n(avec seuil optimal par méthode du coude)',
              fontsize=13, fontweight='bold')

ax3 = axes[1]
ax3.plot(thresholds_valid, distances, color='green', linewidth=2, label='Distance à la droite')
ax3.axvline(optimal_threshold, color='purple', linestyle=':', linewidth=2.5,
            label=f'Coude détecté: {optimal_threshold:.3f}')
ax3.scatter([optimal_threshold], [distances[elbow_idx]], color='purple', s=120, zorder=5)
ax3.set_xlabel("Seuil de Score d'Isolation", fontsize=11)
ax3.set_ylabel('Distance à la droite (normalisée)', fontsize=11)
ax3.set_title('Méthode du Coude : Distance maximale\nà la droite reliant les extrêmes',
              fontsize=13, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('isolation_forest_elbow_threshold.png', dpi=300, bbox_inches='tight')
print("✓ Graphique sauvegardé: isolation_forest_elbow_threshold.png")
plt.show()

print(f"\n{'='*60}")
print(f"SEUIL RETENU : {optimal_threshold:.4f}")
print(f"  → On conserve {optimal_coverage:.1f}% des données de test")
print(f"  → RMSE filtrée : {optimal_rmse:.4f} (vs {rmse_base:.4f} sans filtre)")
print(f"{'='*60}")

## Stratégie d'imputation et robustesse aux valeurs manquantes

In [ ]:
print("=" * 60)
print("STRATÉGIE D'IMPUTATION DES FEATURES")
print("=" * 60)

numeric_features_for_imp = [col for col in feature_cols if not col.endswith('_encoded')]
categorical_features_for_imp = [col for col in feature_cols if col.endswith('_encoded')]

print("""
STRATÉGIE CHOISIE PAR TYPE DE FEATURE :
─────────────────────────────────────────────────────
 Variables numériques  → Imputation par la MÉDIANE
   Robuste aux outliers (salaires, âge, rating...)
   Préserve la distribution centrale
   Insensible aux valeurs extrêmes détectées par IF

 Variables catégorielles encodées → Imputation par le MODE
   La catégorie la plus fréquente est la plus représentative
   Cohérence avec les valeurs entières du LabelEncoder
   Évite d'introduire des catégories inexistantes

 Les imputers sont appris UNIQUEMENT sur X_train
   Pas de data leakage vers le test set
""")

print(f"Features numériques  → MÉDIANE : {numeric_features_for_imp}")
print(f"Features catégorielles → MODE  : {categorical_features_for_imp}")

# Création et fit des imputers
imputer_median = SimpleImputer(strategy='median')
imputer_mode   = SimpleImputer(strategy='most_frequent')

X_train_raw = X.loc[X_train.index]
X_test_raw  = X.loc[X_test.index]

X_train_num = X_train_raw[numeric_features_for_imp]
X_train_cat = X_train_raw[categorical_features_for_imp]
X_test_num  = X_test_raw[numeric_features_for_imp]
X_test_cat  = X_test_raw[categorical_features_for_imp]

imputer_median.fit(X_train_num)
imputer_mode.fit(X_train_cat)

print("\n✓ Imputers entraînés sur X_train uniquement")
print(f"  - Médianes apprises : {dict(zip(numeric_features_for_imp, imputer_median.statistics_.round(2)))}")
print(f"  - Modes appris      : {dict(zip(categorical_features_for_imp, imputer_mode.statistics_))}")

In [ ]:
print("=" * 60)
print("ANALYSE DE ROBUSTESSE : IMPACT DES VALEURS MANQUANTES")
print("=" * 60)

N_REPEATS = 5
N_MAX_RATIO = 0.5
STEP = 10

n_test_samples = X_test_raw.shape[0]
n_max_missing  = int(n_test_samples * N_MAX_RATIO)
n_missing_range = range(STEP, n_max_missing + STEP, STEP)

# RMSE de référence (sans valeurs manquantes, avec imputation)
X_num_ref = imputer_median.transform(X_test_num)
X_cat_ref = imputer_mode.transform(X_test_cat)
X_combined_ref = np.hstack([X_num_ref, X_cat_ref])
X_df_ref = pd.DataFrame(X_combined_ref,
                         columns=numeric_features_for_imp + categorical_features_for_imp,
                         index=X_test_num.index)
X_reordered_ref = X_df_ref[feature_cols]
X_scaled_ref = scaler.transform(X_reordered_ref)
y_pred_ref = best_model_rf.predict(X_scaled_ref)
rmse_base_imp = np.sqrt(mean_squared_error(y_test, y_pred_ref))
print(f"RMSE de référence (sans valeurs manquantes) : {rmse_base_imp:.4f}")

def predict_with_imputation(X_test_num_c, X_test_cat_c):
    X_num_imp = imputer_median.transform(X_test_num_c)
    X_cat_imp = imputer_mode.transform(X_test_cat_c)
    X_combined = np.hstack([X_num_imp, X_cat_imp])
    X_df = pd.DataFrame(X_combined,
                        columns=numeric_features_for_imp + categorical_features_for_imp,
                        index=X_test_num_c.index)
    X_reordered = X_df[feature_cols]
    X_scaled_imp = scaler.transform(X_reordered)
    return best_model_rf.predict(X_scaled_imp)

all_results = {}

for feat in feature_cols:
    rmse_curve = []
    deg_curve  = []
    is_numeric = feat in numeric_features_for_imp

    for n_missing in n_missing_range:
        rmse_vals = []
        for _ in range(N_REPEATS):
            X_test_num_c = X_test_num.copy()
            X_test_cat_c = X_test_cat.copy()
            idx_corrupt  = np.random.choice(n_test_samples, n_missing, replace=False)

            if is_numeric:
                col_idx = X_test_num_c.columns.tolist().index(feat)
                X_test_num_c.iloc[idx_corrupt, col_idx] = np.nan
            else:
                col_idx = X_test_cat_c.columns.tolist().index(feat)
                X_test_cat_c.iloc[idx_corrupt, col_idx] = np.nan

            y_pred_imp = predict_with_imputation(X_test_num_c, X_test_cat_c)
            rmse_vals.append(np.sqrt(mean_squared_error(y_test, y_pred_imp)))

        mean_rmse = np.mean(rmse_vals)
        rmse_curve.append(mean_rmse)
        deg_curve.append((mean_rmse - rmse_base_imp) / rmse_base_imp * 100)

    all_results[feat] = {'rmse': rmse_curve, 'degradation': deg_curve}
    print(f"  ✓ {feat:40s} | RMSE max: {max(rmse_curve):.4f} | Dégradation max: {max(deg_curve):+.2f}%")

print("\n✓ Analyse terminée pour toutes les features!")

In [ ]:
n_features   = len(feature_cols)
n_cols_plot  = 3
n_rows_plot  = math.ceil(n_features / n_cols_plot)
x_axis       = list(n_missing_range)

fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                          figsize=(6 * n_cols_plot, 5 * n_rows_plot))
axes = axes.ravel()

for idx, feat in enumerate(feature_cols):
    ax = axes[idx]
    rmse_curve = all_results[feat]['rmse']
    ax.plot(x_axis, rmse_curve, marker='o', markersize=4,
            color='steelblue', linewidth=2, label='RMSE')
    ax.axhline(rmse_base_imp, color='red', linestyle='--', linewidth=1.5,
               label=f'Référence: {rmse_base_imp:.3f}')
    ax.fill_between(x_axis, rmse_base_imp, rmse_curve, alpha=0.15, color='orange')
    ax.set_xlabel('Nombre de valeurs manquantes', fontsize=9)
    ax.set_ylabel('RMSE', fontsize=9)
    strat = 'médiane' if feat in numeric_features_for_imp else 'mode'
    ax.set_title(f'{feat}\n(imputation: {strat})', fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Impact de l'Imputation sur la RMSE - Toutes les Features",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('robustness_rmse_all_features.png', dpi=200, bbox_inches='tight')
print("✓ Graphique sauvegardé: robustness_rmse_all_features.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                          figsize=(6 * n_cols_plot, 5 * n_rows_plot))
axes = axes.ravel()

for idx, feat in enumerate(feature_cols):
    ax = axes[idx]
    deg_curve = all_results[feat]['degradation']
    ax.plot(x_axis, deg_curve, marker='o', markersize=4,
            color='red', linewidth=2)
    ax.axhline(0, color='black', linestyle='--', linewidth=1.5, label='Référence (0%)')
    ax.fill_between(x_axis, 0, deg_curve, alpha=0.15, color='red')
    ax.set_xlabel('Nombre de valeurs manquantes', fontsize=9)
    ax.set_ylabel('Dégradation RMSE (%)', fontsize=9)
    strat = 'médiane' if feat in numeric_features_for_imp else 'mode'
    ax.set_title(f'{feat}\n(imputation: {strat})', fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Dégradation de la RMSE (%) par Feature - Toutes les Features",
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('robustness_degradation_all_features.png', dpi=200, bbox_inches='tight')
print("✓ Graphique sauvegardé: robustness_degradation_all_features.png")
plt.show()

In [ ]:
print("=" * 70)
print("RÉSUMÉ FINAL - SENSIBILITÉ AUX VALEURS MANQUANTES PAR FEATURE")
print("=" * 70)

summary_rows = []
for feat in feature_cols:
    rmse_curve = all_results[feat]['rmse']
    deg_curve  = all_results[feat]['degradation']
    strat = 'médiane' if feat in numeric_features_for_imp else 'mode'
    summary_rows.append({
        'Feature': feat,
        'Stratégie': strat,
        'RMSE_ref': round(rmse_base_imp, 4),
        'RMSE_max': round(max(rmse_curve), 4),
        'Dégradation_max (%)': round(max(deg_curve), 2),
        'Dégradation_à_50pct (%)': round(deg_curve[-1], 2)
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Dégradation_max (%)', ascending=False)
print(summary_df.to_string(index=False))

most_sensitive  = summary_df.iloc[0]['Feature']
least_sensitive = summary_df.iloc[-1]['Feature']
print(f"\n  → Feature la PLUS sensible  : {most_sensitive}")
print(f"     ({summary_df.iloc[0]['Dégradation_max (%)']:.2f}% de dégradation max)")
print(f"  → Feature la MOINS sensible : {least_sensitive}")
print(f"     ({summary_df.iloc[-1]['Dégradation_max (%)']:.2f}% de dégradation max)")